In [1]:
!pip install datasets
!pip install transformers
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is i

In [2]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
from jiwer import wer


In [4]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [5]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from utils import transcribe_audio

In [9]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from pgd import pgd_attack

In [34]:
example = librispeech[0]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]  # Ground truth transcription
target_transcription = "HELLO WORLD"  # Target transcription

# Run PGD attack
adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription = pgd_attack(
    audio_array=audio_array,
    ground_truth=ground_truth,
    target_transcription=target_transcription,
    model=model,
    processor=processor,
    epsilon=0.05,
    alpha=0.005,
    num_iter=10
)

# Print results
print(f"Ground Truth: {ground_truth}")
print(f"Target Transcription: {target_transcription}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"WER (Ground Truth): {ground_truth_wer:.2f}")
print(f"WER (Target): {target_wer:.2f}")

Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Target Transcription: HELLO WORLD
Adversarial Transcription: AND WHAT SORT OF EVIDENCE AS LOGILY POSSI
WER (Ground Truth): 0.38
WER (Target): 4.00


In [35]:
import IPython.display as ipd
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
adversarial_audio = adversarial_waveform
print("Adversarial Audio:")
display(ipd.Audio(adversarial_audio, rate=16000))

Original Audio:


Adversarial Audio:


In [40]:
from statistics import mean
import json


subset = librispeech.select(range(10))
epsilon_values = [0.01, 0.02, 0.05, 0.1, 0.2]
alpha_values = [0.001, 0.002, 0.005, 0.01, 0.02]
# Define target transcription
target_transcription = "HELLO WORLD"

# Store results
results = []
wer_by_epsilon = {eps: {"ground_truth_wer": [], "target_wer": []} for eps in epsilon_values}

# Select three samples for audio playback (e.g., indices 0, 1, 2)
audio_samples_to_play = [0, 1, 2]  # Adjust if you want different indices
audio_results = {idx: {eps: {} for eps in epsilon_values} for idx in audio_samples_to_play}

# Loop over the dataset
for idx, example in enumerate(subset):
    audio_array = example["audio"]["array"]  # Raw audio waveform
    ground_truth = example["true_text"]  # Ground truth transcription

    print(f"\nSample {idx}")

    # Loop over epsilon values
    for eps_idx, epsilon in enumerate(epsilon_values):
        alpha = alpha_values[eps_idx]
        adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription = pgd_attack(
            audio_array=audio_array,
            ground_truth=ground_truth,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=epsilon,
            alpha=alpha,
            num_iter=10
        )

        # Store result (exclude waveform to avoid JSON serialization issue)
        result = {
            "sample_idx": idx,
            "epsilon": epsilon,
            "ground_truth": ground_truth,
            "adversarial_transcription": adversarial_transcription,
            "ground_truth_wer": ground_truth_wer,
            "target_wer": target_wer
        }
        results.append(result)

        # Collect WERs for averaging
        wer_by_epsilon[epsilon]["ground_truth_wer"].append(ground_truth_wer)
        wer_by_epsilon[epsilon]["target_wer"].append(target_wer)

        # Store audio results for playback if sample is selected
        if idx in audio_samples_to_play:
            audio_results[idx][epsilon] = {
                "adversarial_waveform": adversarial_waveform,
                "ground_truth": ground_truth,
                "adversarial_transcription": adversarial_transcription,
                "ground_truth_wer": ground_truth_wer,
                "target_wer": target_wer
            }

# Play adversarial waveforms for selected samples
print("\nPlaying Adversarial Waveforms for Selected Samples:")
for idx in audio_samples_to_play:
    print(f"\nSample {idx}:")
    for epsilon in epsilon_values:
        audio_data = audio_results[idx][epsilon]
        print(f"\nEpsilon: {epsilon}")
        print(f"Ground Truth: {audio_data['ground_truth']}")
        print(f"Adversarial Transcription: {audio_data['adversarial_transcription']}")
        print(f"WER (Ground Truth): {audio_data['ground_truth_wer']:.2f}")
        print(f"WER (Target): {audio_data['target_wer']:.2f}")
        print("Playing Adversarial Audio:")
        display(ipd.Audio(audio_data['adversarial_waveform'], rate=16000))

# Compute and print average WER for each epsilon
print("\nAverage WER Across All Samples:")
for epsilon in epsilon_values:
    avg_ground_truth_wer = mean(wer_by_epsilon[epsilon]["ground_truth_wer"])
    avg_target_wer = mean(wer_by_epsilon[epsilon]["target_wer"])
    print(f"Epsilon: {epsilon}")
    print(f"  Average Ground Truth WER: {avg_ground_truth_wer:.2f}")
    print(f"  Average Target WER: {avg_target_wer:.2f}")

# Save results to JSON
with open("pgd_results.json", "w") as f:
    json.dump(results, f, indent=2)


Sample 0

Sample 1

Sample 2

Sample 3

Sample 4

Sample 5

Sample 6

Sample 7

Sample 8

Sample 9

Playing Adversarial Waveforms for Selected Samples:

Sample 0:

Epsilon: 0.01
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICLLY POSSIBLE
WER (Ground Truth): 0.12
WER (Target): 4.00
Playing Adversarial Audio:



Epsilon: 0.02
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICIPOSSLE
WER (Ground Truth): 0.25
WER (Target): 3.50
Playing Adversarial Audio:



Epsilon: 0.05
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE AS LOGILY POSSI
WER (Ground Truth): 0.38
WER (Target): 4.00
Playing Adversarial Audio:



Epsilon: 0.1
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENT EBLOGRIPOS
WER (Ground Truth): 0.50
WER (Target): 3.00
Playing Adversarial Audio:



Epsilon: 0.2
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: MEN WHAT SORT OF LOGIPORSE
WER (Ground Truth): 0.62
WER (Target): 2.50
Playing Adversarial Audio:


...


Epsilon: 0.02
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN IFERENCE WARN'T IT
WER (Ground Truth): 0.30
WER (Target): 5.00
Playing Adversarial Audio:



Epsilon: 0.05
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN INFENCE WAR
WER (Ground Truth): 0.30
WER (Target): 4.50
Playing Adversarial Audio:



Epsilon: 0.1
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN ICE RON
WER (Ground Truth): 0.30
WER (Target): 4.50
Playing Adversarial Audio:



Epsilon: 0.2
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: INO P WARNF
WER (Ground Truth): 1.00
WER (Target): 1.50
Playing Adversarial Audio:



Average WER Across All Samples:
Epsilon: 0.01
  Average Ground Truth WER: 0.24
  Average Target WER: 6.00
Epsilon: 0.02
  Average Ground Truth WER: 0.28
  Average Target WER: 5.85
Epsilon: 0.05
  Average Ground Truth WER: 0.45
  Average Target WER: 5.40
Epsilon: 0.1
  Average Ground Truth WER: 0.59
  Average Target WER: 4.80
Epsilon: 0.2
  Average Ground Truth WER: 0.79
  Average Target WER: 3.25


In [39]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [41]:
!cp /content/pgd_results.json /content/drive/MyDrive/pgd_results.json

In [42]:

from google.colab import files
files.download('pgd_results.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>